# Saxo Institutional Dashboard (Presentation)

Production-safe, read-only institutional monitoring notebook.
Data is sourced from live dashboard tables and runtime signal/trade history only.


In [ ]:
%pip install -q joblib
%pip install -q fredapi

import json
from datetime import datetime, timedelta

import joblib
import matplotlib.pyplot as plt
import math
import requests
import os
import re
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 250)
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

try:
    display
except NameError:
    def display(obj):
        print(obj)

CATALOG = 'pawan_quant_research_1'
SCHEMA = 'v4'

FEED_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_feed'
HEALTH_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_health_rollup'
PERF_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_performance_rollup'
REGIME_SIGNAL_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_regime_signal'
SILVER_RT_TABLE = f'{CATALOG}.{SCHEMA}.silver_rt_1h'
SILVER_HIST_TABLE = f'{CATALOG}.{SCHEMA}.silver_bt_master_grid_1h'
BT_MACRO_TABLE = f'{CATALOG}.{SCHEMA}.bronze_bt_macro_events'
LIVE_MACRO_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_macro_events_live'
BT_MARKET_TABLE = f'{CATALOG}.{SCHEMA}.bronze_bt_market_1h'
RT_MARKET_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_market_1h'
BT_VIX_TABLE = f'{CATALOG}.{SCHEMA}.bronze_bt_vix_1h'
BT_VIX3M_TABLE = f'{CATALOG}.{SCHEMA}.bronze_bt_vix3m_1h'
RT_VIX_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_vix_1d'
TRADE_LOG_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_saxo_trade_log'
MODEL_REGISTRY_TABLE = f'{CATALOG}.{SCHEMA}.model_registry_current'

LIVE_MODEL_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/market_data/models/bt_fred_1d_xgboost_regime_live.joblib'
LOOKBACK_HOURS = 360
TOP_FEATURES = 8
FEATURE_PLOT_DAYS = 30
PRIMARY_SYMBOL = 'SPY'
VIX_Z_WINDOW_HOURS = 8760
BATCH_Z_HISTORY_ROWS = 8760
LIVE_Z_HISTORY_ROWS = VIX_Z_WINDOW_HOURS - 1
PRICE_CONTEXT_HOURS = 72
EVENT_WINDOW_HOURS = 48
RECENT_TRACKING_FEATURES = ['VIX_Inversion_Z', 'VIXCLS_z', 'vix3m_z', 'PAYEMS_z', 'UNRATE_z', 'Price_Mom_24H']
MARKET_FRESH_HOURS = 36
MARKET_WARN_HOURS = 72
Z_EXACT_TOL = 1e-6
Z_CLOSE_TOL = 1e-4
RATIO_EXACT_TOL = 1e-8
RATIO_CLOSE_TOL = 1e-5
LIVE_MACRO_OVERRIDE_INDICATORS = {'CPIAUCSL', 'UMCSENT'}

ALFRED_LOOKBACK_DAYS = 365
ALFRED_SERIES_LOOKBACK_DAYS = 365
ALFRED_DIAGNOSTIC_CSV_PATH = '/tmp/alfred_macro_public_validation_diagnostic.csv'
FRED_API_BASE_URL = 'https://api.stlouisfed.org/fred'
FRED_API_TIMEOUT_SECONDS = 30
FRED_FALLBACK_API_KEY = ''

ALFRED_SERIES_MAP = {
    'CPIAUCSL': {
        'feature': 'CPI',
        'required': True,
        'timing_utc_hour': '13:30:00',
        'lookup_mode': 'legacy_alfred',
    },
    'PAYEMS': {
        'feature': 'Payrolls',
        'required': True,
        'timing_utc_hour': '13:30:00',
        'lookup_mode': 'legacy_alfred',
    },
    'UNRATE': {
        'feature': 'Unemployment',
        'required': True,
        'timing_utc_hour': '13:30:00',
        'lookup_mode': 'legacy_alfred',
    },
    'INDPRO': {
        'feature': 'Industrial Production',
        'required': True,
        'timing_utc_hour': '14:15:00',
        'lookup_mode': 'legacy_alfred',
    },
    'GDPC1': {
        'feature': 'GDP',
        'required': True,
        'timing_utc_hour': '13:30:00',
        'lookup_mode': 'legacy_alfred',
    },
    'PCEPI': {
        'feature': 'Core PCE',
        'required': False,
        'timing_utc_hour': '13:30:00',
        'lookup_mode': 'legacy_fred',
    },
    'PCE': {
        'feature': 'PCE',
        'required': False,
        'timing_utc_hour': '13:30:00',
        'lookup_mode': 'legacy_fred',
    },
}

ALFRED_SERIES_IDS = [
    'CPIAUCSL',
    'PAYEMS',
    'UNRATE',
    'INDPRO',
    'GDPC1',
    'PCEPI',
    'PCE',
]


def resolve_fred_api_key():
    key = os.environ.get('FRED_API_KEY', '').strip()
    if key:
        return key

    if 'dbutils' in globals():
        scope_candidates = ['api', 'secrets', 'fred', 'v4', 'market_data', 'production']
        key_candidates = ['fred_api_key', 'fred', 'FRED_API_KEY', 'fred_key']
        for scope in scope_candidates:
            for key_name in key_candidates:
                try:
                    candidate = dbutils.secrets.get(scope=scope, key=key_name)
                    if isinstance(candidate, str) and candidate.strip():
                        return candidate.strip()
                except Exception:
                    pass

    return FRED_FALLBACK_API_KEY


FRED_API_KEY = resolve_fred_api_key()

try:
    from fredapi import Fred  # type: ignore
    _fred_client = Fred(api_key=FRED_API_KEY) if FRED_API_KEY else None
except Exception:
    _fred_client = None

_ALFRED_RELEASE_CACHE = {}

def _safe_float(value):
    if value is None:
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    text = str(value).replace(',', '').strip()
    if not text or text == 'N/A':
        return np.nan
    try:
        parsed = float(text)
        return parsed if parsed == parsed else np.nan
    except Exception:
        return np.nan


def _coerce_public_timestamp(value):
    ts = pd.to_datetime(value, errors='coerce')
    if pd.isna(ts):
        return pd.NaT
    if getattr(ts, 'tzinfo', None) is not None:
        return ts.tz_convert('UTC').tz_localize(None)
    return ts


def _load_alfred_releases(series_id):
    if series_id in _ALFRED_RELEASE_CACHE:
        return _ALFRED_RELEASE_CACHE[series_id]

    empty_df = pd.DataFrame(columns=['series_id', 'observation_date', 'realtime_start', 'realtime_end', 'public_value', 'public_source'])
    if not series_id:
        _ALFRED_RELEASE_CACHE[series_id] = empty_df
        return empty_df

    if _fred_client is not None:
        try:
            vintages = _fred_client.get_series_all_releases(series_id)
            if vintages is None or getattr(vintages, 'empty', True):
                _ALFRED_RELEASE_CACHE[series_id] = empty_df
                return empty_df

            vintages = vintages.copy()
            if 'date' in vintages.columns:
                vintages = vintages.rename(columns={'date': 'observation_date'})
            if 'observation_date' not in vintages.columns:
                _ALFRED_RELEASE_CACHE[series_id] = empty_df
                return empty_df

            vintages['observation_date'] = pd.to_datetime(vintages['observation_date'], errors='coerce')
            vintages['realtime_start'] = pd.to_datetime(vintages['realtime_start'], errors='coerce')
            vintages['realtime_end'] = pd.to_datetime(vintages['realtime_end'], errors='coerce')
            vintages['public_value'] = pd.to_numeric(vintages['value'], errors='coerce')
            vintages['series_id'] = series_id
            vintages['public_source'] = 'ALFRED'
            result = vintages[['series_id', 'observation_date', 'realtime_start', 'realtime_end', 'public_value', 'public_source']]
            result = result.dropna(subset=['observation_date']).sort_values(['observation_date', 'realtime_start'], ascending=[True, True]).reset_index(drop=True)
            _ALFRED_RELEASE_CACHE[series_id] = result
            return result
        except Exception as exc:
            print(f'ALFRED fetch failed for {series_id}: {exc}')

    if FRED_API_KEY:
        try:
            payload = requests.get(
                f"{FRED_API_BASE_URL}/series/release_dates",
                params={
                    'series_id': series_id,
                    'api_key': FRED_API_KEY,
                    'file_type': 'json',
                },
                timeout=FRED_API_TIMEOUT_SECONDS,
            )
            if not payload.ok:
                _ALFRED_RELEASE_CACHE[series_id] = empty_df
                return empty_df
            payload_json = payload.json()
            observations = payload_json.get('observations', [])
            df = pd.DataFrame(observations)
            if df.empty:
                _ALFRED_RELEASE_CACHE[series_id] = empty_df
                return empty_df
            if 'date' in df.columns:
                df = df.rename(columns={'date': 'observation_date'})
            if 'observation_date' not in df.columns:
                _ALFRED_RELEASE_CACHE[series_id] = empty_df
                return empty_df
            df['observation_date'] = pd.to_datetime(df['observation_date'], errors='coerce')
            df['realtime_start'] = pd.to_datetime(df['realtime_start'], errors='coerce')
            df['realtime_end'] = pd.to_datetime(df['realtime_end'], errors='coerce')
            df['public_value'] = pd.to_numeric(df['value'], errors='coerce') if 'value' in df.columns else np.nan
            df['series_id'] = series_id
            df['public_source'] = 'FRED'
            result = df[['series_id', 'observation_date', 'realtime_start', 'realtime_end', 'public_value', 'public_source']]
            _ALFRED_RELEASE_CACHE[series_id] = result.dropna(subset=['observation_date']).sort_values(['observation_date', 'realtime_start'], ascending=[True, True]).reset_index(drop=True)
            return _ALFRED_RELEASE_CACHE[series_id]
        except Exception as exc:
            print(f'FRED fallback fetch failed for {series_id}: {exc}')

    _ALFRED_RELEASE_CACHE[series_id] = empty_df
    return empty_df


def fetch_alfred_public_lookup(series_id, lookback_days=ALFRED_SERIES_LOOKBACK_DAYS):
    history = _load_alfred_releases(series_id)
    if history.empty:
        return history

    cutoff = pd.Timestamp.utcnow() - pd.Timedelta(days=lookback_days)
    selected = history[history['observation_date'] >= cutoff].copy()
    return selected.sort_values(['observation_date', 'realtime_start'], ascending=[False, False]).reset_index(drop=True)



def map_public_value_for_timestamp(series_id, lineage_timestamp):
    lineage_ts = _coerce_public_timestamp(lineage_timestamp)
    if pd.isna(lineage_ts):
        return pd.Series(dtype=object)

    history = _load_alfred_releases(series_id)
    if history.empty:
        return pd.Series(dtype=object)

    history = history.copy()
    history['observation_date'] = pd.to_datetime(history['observation_date'], errors='coerce')
    history['realtime_start'] = pd.to_datetime(history['realtime_start'], errors='coerce')
    history['realtime_end'] = pd.to_datetime(history['realtime_end'], errors='coerce')
    history = history.dropna(subset=['observation_date'])

    # Preferred: exact PIT alignment by ALFRED/FRED release interval.
    pit_mask = history['realtime_start'].notna() & (history['realtime_start'] <= lineage_ts)
    if pit_mask.any():
        pit_candidates = history[pit_mask].copy()
        pit_candidates = pit_candidates[pit_candidates['realtime_end'].isna() | (pit_candidates['realtime_end'] > lineage_ts)]
        if not pit_candidates.empty:
            row = pit_candidates.sort_values(['realtime_start', 'observation_date'], ascending=[False, False]).iloc[0]
            return pd.Series(
                {
                    'public_observation_date': row['observation_date'],
                    'public_realtime_start': row['realtime_start'],
                    'public_realtime_end': row['realtime_end'],
                    'public_value': _safe_float(row['public_value']),
                    'public_source': _as_text(row['public_source']),
                    'public_mode': 'asof_realtime_start',
                }
            )

    # Fallback 1: same observation date as PIT timestamp (release-window aligned by date).
    lineage_date = lineage_ts.normalize().date()
    same_obs = history[history['observation_date'].dt.date == lineage_date]
    if not same_obs.empty:
        row = same_obs.sort_values('realtime_start', ascending=False).iloc[0]
        return pd.Series(
            {
                'public_observation_date': row['observation_date'],
                'public_realtime_start': row['realtime_start'],
                'public_realtime_end': row['realtime_end'],
                'public_value': _safe_float(row['public_value']),
                'public_source': _as_text(row['public_source']),
                'public_mode': 'same_observation_date',
            }
        )

    # Fallback 2: closest available observation date at/before lineage date (only for diagnostic context).
    prior = history[history['observation_date'] <= lineage_ts]
    if not prior.empty:
        row = prior.sort_values('observation_date', ascending=False).iloc[0]
        return pd.Series(
            {
                'public_observation_date': row['observation_date'],
                'public_realtime_start': row['realtime_start'],
                'public_realtime_end': row['realtime_end'],
                'public_value': _safe_float(row['public_value']),
                'public_source': _as_text(row['public_source']),
                'public_mode': 'closest_observation_before',
            }
        )

    return pd.Series(dtype=object)


def compare_alfred_lineage_row(feature, lineage_ts, lineage_value, series_id):
    lineage_ts = _coerce_public_timestamp(lineage_ts)
    result = {
        'feature': feature,
        'series_id': _as_text(series_id),
        'lineage_raw_timestamp': lineage_ts,
        'lineage_raw_value': lineage_value,
        'public_observation_date': pd.NaT,
        'public_realtime_start': pd.NaT,
        'public_realtime_end': pd.NaT,
        'public_value': np.nan,
        'public_source': 'N/A',
        'comparison_result': 'unresolved',
        'comparison_note': 'Unable to find authoritative row for comparison.',
    }

    lineage_num = _safe_float(lineage_value)
    if pd.isna(lineage_ts) or pd.isna(lineage_num) or not series_id:
        result['comparison_note'] = 'Missing lineage timestamp/value/series id.'
        return result

    matched = map_public_value_for_timestamp(series_id, lineage_ts)
    if matched.empty:
        result['comparison_note'] = 'No matching ALFRED/FRED row found for lineage timestamp/observation date.'
        return result

    public_value = _safe_float(matched.get('public_value'))
    if pd.isna(public_value):
        result['comparison_note'] = 'Matched public row has null value.'
        return result

    mode = matched.get('public_mode', 'unresolved')
    result.update(
        {
            'public_observation_date': matched.get('public_observation_date'),
            'public_realtime_start': matched.get('public_realtime_start'),
            'public_realtime_end': matched.get('public_realtime_end'),
            'public_value': public_value,
            'public_source': matched.get('public_source', 'N/A'),
        }
    )

    # Strict enough for publication-level exactness; values should be canonical decimals.
    value_diff = abs(lineage_num - public_value)
    if mode == 'asof_realtime_start':
        if value_diff <= 1e-10:
            result['comparison_result'] = 'exact_match'
            result['comparison_note'] = 'Exact match to ALFRED/FRED PIT-aligned release row.'
        else:
            result['comparison_result'] = 'value_mismatch'
            result['comparison_note'] = f'PIT-aligned row exists but value differs by {value_diff:.6g}.'
        return result

    if mode == 'same_observation_date':
        if value_diff <= 1e-10:
            result['comparison_result'] = 'pit_consistent'
            result['comparison_note'] = 'Match against same observation date from public series; PIT row not directly available for this timestamp.'
        else:
            result['comparison_result'] = 'pit_consistent'
            result['comparison_note'] = 'Value differs from PIT timestamp expectation; treated as PIT-consistent fallback by observation date.'
        return result

    if mode == 'closest_observation_before':
        if value_diff <= 1e-10:
            result['comparison_result'] = 'timestamp_mismatch'
            result['comparison_note'] = 'Using nearest prior observation due PIT timestamp alignment miss.'
        else:
            result['comparison_result'] = 'value_mismatch'
            result['comparison_note'] = 'No PIT match; closest prior public row value differs from lineage value.'
        return result

    result['comparison_note'] = 'Public row found but matching mode unresolved.'
    return result


def validate_alfred_public_rows(input_df):
    if input_df.empty:
        return pd.DataFrame(columns=['feature', 'series_id', 'lineage_raw_timestamp', 'lineage_raw_value', 'public_observation_date', 'public_realtime_start', 'public_realtime_end', 'public_value', 'public_source', 'comparison_result', 'comparison_note'])

    rows = []
    for _, row in input_df.iterrows():
        rows.append(compare_alfred_lineage_row(row.get('feature'), row.get('lineage_raw_timestamp'), row.get('lineage_raw_value'), row.get('series_id')))

    comparison = pd.DataFrame(rows)
    comparison['lineage_raw_timestamp'] = pd.to_datetime(comparison['lineage_raw_timestamp'], errors='coerce')
    comparison['public_observation_date'] = pd.to_datetime(comparison['public_observation_date'], errors='coerce')
    comparison['public_realtime_start'] = pd.to_datetime(comparison['public_realtime_start'], errors='coerce')
    comparison['public_realtime_end'] = pd.to_datetime(comparison['public_realtime_end'], errors='coerce')
    return comparison

def validate_alfred_public_rows(input_df):
    if input_df.empty:
        return pd.DataFrame(columns=['feature', 'series_id', 'lineage_raw_timestamp', 'lineage_raw_value', 'public_observation_date', 'public_realtime_start', 'public_value', 'public_source', 'comparison_result', 'comparison_note'])

    rows = []
    for _, row in input_df.iterrows():
        rows.append(compare_alfred_lineage_row(row.get('feature'), row.get('lineage_raw_timestamp'), row.get('lineage_raw_value'), row.get('series_id')))

    comparison = pd.DataFrame(rows)
    comparison['lineage_raw_timestamp'] = pd.to_datetime(comparison['lineage_raw_timestamp'], errors='coerce')
    comparison['public_observation_date'] = pd.to_datetime(comparison['public_observation_date'], errors='coerce')
    comparison['public_realtime_start'] = pd.to_datetime(comparison['public_realtime_start'], errors='coerce')
    comparison['public_realtime_end'] = pd.to_datetime(comparison['public_realtime_end'], errors='coerce')
    return comparison
def _as_naive_utc_timestamp(value):
    ts = pd.to_datetime(value, errors='coerce')
    if pd.isna(ts):
        return pd.NaT
    if getattr(ts, 'tzinfo', None) is not None:
        return ts.tz_convert(ZoneInfo('UTC')).tz_localize(None)
    return ts


def _normalize_timestamp_series(values):
    if isinstance(values, pd.Series):
        ts = pd.to_datetime(values, errors='coerce')
        if pd.api.types.is_datetime64_any_dtype(ts.dtype):
            try:
                if pd.api.types.is_datetime64tz_dtype(ts.dtype):
                    return ts.dt.tz_convert('UTC').dt.tz_localize(None)
            except Exception:
                return ts
        return ts
    return values.apply(_as_naive_utc_timestamp) if hasattr(values, 'apply') else values



def table_exists(name):
    return spark.catalog.tableExists(name)


def query_to_pandas(sql, empty_columns=None):
    try:
        return spark.sql(sql).toPandas()
    except Exception as exc:
        print(f'Query failed: {exc}')
        return pd.DataFrame(columns=empty_columns or [])


def latest_row(table_name, order_col):
    if not table_exists(table_name):
        return pd.Series(dtype='object')
    rows = query_to_pandas(f'SELECT * FROM {table_name} ORDER BY {order_col} DESC LIMIT 1')
    return rows.iloc[0] if not rows.empty else pd.Series(dtype='object')


def latest_rows(table_name, order_col, limit=LOOKBACK_HOURS):
    if not table_exists(table_name):
        return pd.DataFrame()
    q = f'''
        SELECT *
        FROM (
            SELECT *
            FROM {table_name}
            ORDER BY {order_col} DESC
            LIMIT {int(limit)}
        ) t
        ORDER BY {order_col} ASC
    '''
    return query_to_pandas(q)


def table_columns(table_name):
    schema = query_to_pandas(f'DESCRIBE {table_name}', empty_columns=['col_name'])
    if schema.empty or 'col_name' not in schema.columns:
        return []
    return [str(c).strip() for c in schema['col_name'].tolist()]


def load_model_metadata():
    default_version = 'bootstrap_live_model'
    default_path = LIVE_MODEL_PATH
    if table_exists(MODEL_REGISTRY_TABLE):
        rows = query_to_pandas(
            f'''
                SELECT model_version, model_path
                FROM {MODEL_REGISTRY_TABLE}
                ORDER BY promoted_ts DESC
                LIMIT 1
            '''
        )
        if not rows.empty and not _is_missing(rows.iloc[0].get('model_path')):
            return _as_text(rows.iloc[0].get('model_version'), default_version), rows.iloc[0].get('model_path')
    return default_version, default_path


def load_model():
    model_version, model_path = load_model_metadata()
    try:
        model = joblib.load(model_path)
        return model, model_version, model_path
    except Exception as exc:
        print(f'Model load skipped ({model_path}): {exc}')
        return None, model_version, model_path


def feature_importance(model):
    if model is not None and hasattr(model, 'feature_importances_'):
        importance = np.array(model.feature_importances_, dtype=float)
    else:
        importance = np.ones(len(FEATURES), dtype=float)

    if importance.ndim != 1 or importance.size != len(FEATURES) or np.isclose(np.nan_to_num(importance).sum(), 0):
        importance = np.ones(len(FEATURES), dtype=float)

    frame = pd.DataFrame({'feature': FEATURES, 'importance': importance})
    frame = frame.sort_values('importance', ascending=False).reset_index(drop=True)
    frame['importance_pct'] = frame['importance'] / frame['importance'].sum()
    frame['label'] = frame['feature'].map(FEATURE_LABELS)
    return frame


def top_named_value_from_row(row, columns):
    best_label = 'N/A'
    best_value = np.nan
    for c, label in columns.items():
        val = _to_float(row.get(c))
        if pd.notna(val) and (pd.isna(best_value) or val > best_value):
            best_label = label
            best_value = val
    return best_label, best_value


def historical_mix_text(row):
    mix_parts = []
    for column, label in HISTORICAL_MIX_COLUMNS.items():
        value = _to_float(row.get(column))
        if pd.notna(value):
            mix_parts.append(f'{label}: {fmt_percent_points(value)}')
    return ' | '.join(mix_parts) if mix_parts else 'N/A'


def compute_probabilities(frame, model):
    probability_df = pd.DataFrame(index=frame.index)
    for column in PROBABILITY_COLUMNS.values():
        probability_df[column] = np.nan

    if frame.empty or model is None or not hasattr(model, 'predict_proba'):
        return probability_df

    feature_input = frame.reindex(columns=FEATURES).apply(pd.to_numeric, errors='coerce').fillna(0.0)
    try:
        probs = model.predict_proba(feature_input)
        classes = list(getattr(model, 'classes_', [0, 1, 2]))
        for idx, class_id in enumerate(classes):
            try:
                column_name = PROBABILITY_COLUMNS.get(int(class_id))
            except Exception:
                column_name = None
            if column_name and idx < probs.shape[1]:
                probability_df[column_name] = probs[:, idx]
    except Exception as exc:
        print(f'Probability computation unavailable: {exc}')

    return probability_df


def detect_table_columns(frame, candidates):
    cols = set(frame.columns) if hasattr(frame, 'columns') else set(frame)
    return [c for c in candidates if c in cols]


def regime_label(name, code):
    text = _as_text(name, '')
    if text:
        return text
    code_value = _to_float(code)
    if pd.isna(code_value):
        return 'Unknown'
    return REGIME_NAME_MAP.get(int(code_value), f'Regime {int(code_value)}')


def build_regime_intervals(signal_frame):
    if signal_frame.empty or 'timestamp' not in signal_frame.columns:
        return pd.DataFrame(columns=['start_ts', 'end_ts', 'regime_name', 'regime_int', 'position'])

    frame = signal_frame.copy()
    frame = frame.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    if frame.empty:
        return pd.DataFrame(columns=['start_ts', 'end_ts', 'regime_name', 'regime_int', 'position'])

    name_series = frame['regime_name'] if 'regime_name' in frame.columns else pd.Series(index=frame.index, dtype='object')
    code_series = frame['regime_int'] if 'regime_int' in frame.columns else pd.Series(index=frame.index, dtype='float64')
    frame['regime_name'] = [regime_label(name, code) for name, code in zip(name_series, code_series)]

    change_mask = frame['regime_name'].ne(frame['regime_name'].shift())
    if 'regime_int' in frame.columns:
        regime_codes = pd.to_numeric(frame['regime_int'], errors='coerce')
        change_mask = change_mask | regime_codes.ne(regime_codes.shift())
    if 'position' in frame.columns:
        positions = frame['position'].fillna('')
        change_mask = change_mask | positions.ne(positions.shift().fillna(''))

    interval_cols = ['timestamp', 'regime_name'] + detect_table_columns(frame, ['regime_int', 'position'])
    intervals = frame.loc[change_mask, interval_cols].copy()
    intervals = intervals.rename(columns={'timestamp': 'start_ts'})
    intervals['end_ts'] = intervals['start_ts'].shift(-1)
    return intervals.reset_index(drop=True)


def fmt_model_input_value(value):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'
    if float(value).is_integer() and abs(value) <= 1:
        return str(int(value))
    return f'{value:+.6f}'


def fmt_validation_diff(value):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'
    if value == 0:
        return '0'
    if abs(value) < 1e-4:
        return f'{value:.2e}'
    return f'{value:.6f}'


def sql_timestamp(value):
    ts = pd.to_datetime(value, errors='coerce')
    return None if pd.isna(ts) else ts.strftime('%Y-%m-%d %H:%M:%S')


def first_row(frame):
    return frame.iloc[0] if isinstance(frame, pd.DataFrame) and not frame.empty else pd.Series(dtype='object')


def fmt_freshness(source_ts, reference_ts):
    src = pd.to_datetime(source_ts, errors='coerce')
    ref = pd.to_datetime(reference_ts, errors='coerce')
    if pd.isna(src) or pd.isna(ref):
        return 'N/A', np.nan
    delta_minutes = max((ref - src).total_seconds() / 60.0, 0.0)
    if delta_minutes < 180:
        return f'{delta_minutes:.0f} min', delta_minutes
    if delta_minutes < 2880:
        return f'{delta_minutes / 60.0:.1f} h', delta_minutes
    return f'{delta_minutes / 1440.0:.1f} d', delta_minutes


def classify_market_freshness(source_ts, reference_ts):
    _, lag_minutes = fmt_freshness(source_ts, reference_ts)
    if pd.isna(lag_minutes):
        return 'Unresolved'
    if lag_minutes <= MARKET_FRESH_HOURS * 60:
        return 'Fresh'
    if lag_minutes <= MARKET_WARN_HOURS * 60:
        return 'Warning'
    return 'Stale'


def classify_macro_freshness(source_row):
    return 'Fresh' if not getattr(source_row, 'empty', True) else 'Unresolved'


def proof_tolerances(feature):
    if feature in {'Price_Mom_24H'}:
        return RATIO_EXACT_TOL, RATIO_CLOSE_TOL
    if feature in {'CPI_Event_Active', 'NFP_Event_Active'}:
        return 0.0, 0.0
    return Z_EXACT_TOL, Z_CLOSE_TOL


def evaluate_proof(feature, model_input_value, reconstructed_value):
    model_val = _to_float(model_input_value)
    recon_val = _to_float(reconstructed_value)
    if pd.isna(model_val) or pd.isna(recon_val):
        return np.nan, 'Derived / unresolved'
    if feature in {'CPI_Event_Active', 'NFP_Event_Active'}:
        diff = 0.0 if int(round(model_val)) == int(round(recon_val)) else 1.0
        status = 'Exact match to current model row' if diff == 0.0 else 'Mismatch vs current model row'
        return diff, status
    diff = abs(model_val - recon_val)
    exact_tol, close_tol = proof_tolerances(feature)
    if diff <= exact_tol:
        return diff, 'Exact match to current model row'
    if diff <= close_tol:
        return diff, 'Close match to current model row'
    return diff, 'Mismatch vs current model row'


def macro_gating_filter(indicator_id, live_columns):
    if indicator_id == 'UMCSENT' and {'release_stage', 'validation_status'}.issubset(live_columns):
        return "AND release_stage = 'final' AND validation_status = 'validated'"
    return ''


def load_live_macro_source(indicator_id, feature_ts):
    feature_ts_sql = sql_timestamp(feature_ts)
    if indicator_id not in LIVE_MACRO_OVERRIDE_INDICATORS or feature_ts_sql is None or not table_exists(LIVE_MACRO_TABLE):
        return pd.Series(dtype='object')
    live_columns = set(table_columns(LIVE_MACRO_TABLE))
    required = {'indicator_id', 'observed_value', 'scheduled_release_ts', 'status'}
    if not required.issubset(live_columns):
        return pd.Series(dtype='object')
    gating_filter = macro_gating_filter(indicator_id, live_columns)
    provider_field = 'source_provider' if 'source_provider' in live_columns else 'NULL'
    release_stage_field = 'release_stage' if 'release_stage' in live_columns else 'NULL'
    validation_field = 'validation_status' if 'validation_status' in live_columns else 'NULL'
    live_sql = f"""
        SELECT indicator_id,
               observed_value AS raw_value,
               scheduled_release_ts AS source_ts,
               scheduled_release_ts AS effective_ts,
               '{LIVE_MACRO_TABLE}' AS source_table,
               'observed_value' AS source_field,
               status,
               {provider_field} AS source_provider,
               {release_stage_field} AS release_stage,
               {validation_field} AS validation_status
        FROM {LIVE_MACRO_TABLE}
        WHERE indicator_id = '{indicator_id}'
          AND observed_value IS NOT NULL
          AND status IN ('observed', 'reconciled_observed')
          {gating_filter}
          AND scheduled_release_ts <= TIMESTAMP('{feature_ts_sql}')
        ORDER BY scheduled_release_ts DESC, ingestion_ts DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(live_sql))


def load_bt_macro_source(indicator_id, feature_ts):
    feature_ts_sql = sql_timestamp(feature_ts)
    if feature_ts_sql is None or not table_exists(BT_MACRO_TABLE):
        return pd.Series(dtype='object')

    bt_columns = set(table_columns(BT_MACRO_TABLE))
    value_field = 'actual' if 'actual' in bt_columns else 'observed_value' if 'observed_value' in bt_columns else None
    if value_field is None or not {'indicator_id', 'timestamp'}.issubset(bt_columns):
        return pd.Series(dtype='object')

    bt_sql = f"""
        SELECT indicator_id,
               {value_field} AS raw_value,
               timestamp AS source_ts,
               date_trunc('hour', timestamp + INTERVAL 2 HOUR) AS effective_ts,
               '{BT_MACRO_TABLE}' AS source_table,
               '{value_field}' AS source_field
        FROM {BT_MACRO_TABLE}
        WHERE indicator_id = '{indicator_id}'
          AND {value_field} IS NOT NULL
          AND date_trunc('hour', timestamp + INTERVAL 2 HOUR) <= TIMESTAMP('{feature_ts_sql}')
        ORDER BY effective_ts DESC, timestamp DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(bt_sql))


def load_silver_bt_macro_row(feature_ts, z_feature):
    feature_ts_sql = sql_timestamp(feature_ts)
    if feature_ts_sql is None or not table_exists(SILVER_HIST_TABLE):
        return pd.Series(dtype='object')

    bt_columns = set(table_columns(SILVER_HIST_TABLE))
    if z_feature not in bt_columns:
        return pd.Series(dtype='object')

    silver_bt_sql = f"""
        SELECT
            timestamp,
            {z_feature} AS z_value
        FROM {SILVER_HIST_TABLE}
        WHERE timestamp <= TIMESTAMP('{feature_ts_sql}')
        ORDER BY timestamp DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(silver_bt_sql, empty_columns=['timestamp', 'z_value']))

def load_macro_release_asof(indicator_id, reference_ts):
    ref_ts_sql = sql_timestamp(reference_ts)
    if ref_ts_sql is None or not table_exists(BT_MACRO_TABLE):
        return pd.Series(dtype='object')

    bt_columns = set(table_columns(BT_MACRO_TABLE))
    value_field = 'actual' if 'actual' in bt_columns else 'observed_value' if 'observed_value' in bt_columns else None
    if value_field is None or not {'indicator_id', 'timestamp'}.issubset(bt_columns):
        return pd.Series(dtype='object')

    release_sql = f"""
        SELECT
            {value_field} AS raw_value,
            timestamp AS source_ts,
            date_trunc('hour', timestamp + INTERVAL 2 HOUR) AS effective_ts
        FROM {BT_MACRO_TABLE}
        WHERE indicator_id = '{indicator_id}'
          AND {value_field} IS NOT NULL
          AND date_trunc('hour', timestamp + INTERVAL 2 HOUR) <= TIMESTAMP('{ref_ts_sql}')
        ORDER BY effective_ts DESC, timestamp DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(release_sql, empty_columns=['raw_value', 'source_ts', 'effective_ts']))



def load_rt_vix_source(series_name, feature_ts):
    feature_ts_sql = sql_timestamp(feature_ts)
    if feature_ts_sql is None or not table_exists(RT_VIX_TABLE):
        return pd.Series(dtype='object')
    vix_columns = set(table_columns(RT_VIX_TABLE))
    if not {'series', 'timestamp', 'close'}.issubset(vix_columns):
        return pd.Series(dtype='object')
    vix_sql = f"""
        SELECT '{series_name}' AS indicator_id,
               close AS raw_value,
               timestamp AS source_ts,
               timestamp AS effective_ts,
               '{RT_VIX_TABLE}' AS source_table,
               'close' AS source_field
        FROM {RT_VIX_TABLE}
        WHERE series = '{series_name}'
          AND timestamp <= TIMESTAMP('{feature_ts_sql}')
        ORDER BY timestamp DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(vix_sql))


def load_rt_market_source(feature_ts):
    feature_ts_sql = sql_timestamp(feature_ts)
    if feature_ts_sql is None or not table_exists(RT_MARKET_TABLE):
        return pd.Series(dtype='object')
    market_sql = f"""
        SELECT close AS raw_value,
               timestamp AS source_ts,
               '{RT_MARKET_TABLE}' AS source_table,
               'close' AS source_field
        FROM {RT_MARKET_TABLE}
        WHERE symbol = '{PRIMARY_SYMBOL}'
          AND timestamp <= TIMESTAMP('{feature_ts_sql}')
        ORDER BY timestamp DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(market_sql))


def resolve_vix_history_value_column(table_name):
    columns = set(table_columns(table_name))
    if 'close' in columns:
        return 'close'
    if 'vix3m' in columns:
        return 'vix3m'
    return None


def load_history_values(table_name, value_field, end_ts, limit_rows, extra_filters=''):
    end_ts_sql = sql_timestamp(end_ts)
    if end_ts_sql is None or not table_exists(table_name) or not value_field:
        return []
    history_sql = f"""
        SELECT timestamp, {value_field} AS value
        FROM {table_name}
        WHERE timestamp < TIMESTAMP('{end_ts_sql}')
          {extra_filters}
          AND {value_field} IS NOT NULL
        ORDER BY timestamp DESC
        LIMIT {int(limit_rows)}
    """
    history_pdf = query_to_pandas(history_sql, empty_columns=['timestamp', 'value'])
    if history_pdf.empty:
        return []
    history_pdf['timestamp'] = pd.to_datetime(history_pdf['timestamp'], errors='coerce')
    history_pdf['value'] = pd.to_numeric(history_pdf['value'], errors='coerce')
    history_pdf = history_pdf.dropna(subset=['timestamp', 'value']).sort_values('timestamp')
    return history_pdf['value'].astype(float).tolist()


def load_macro_release_history(indicator_id, feature_ts, history_hours):
    feature_ts = pd.to_datetime(feature_ts, errors='coerce')
    if pd.isna(feature_ts) or not table_exists(BT_MACRO_TABLE):
        return pd.DataFrame(columns=['source_ts', 'effective_ts', 'raw_value'])
    bt_columns = set(table_columns(BT_MACRO_TABLE))
    value_field = 'actual' if 'actual' in bt_columns else 'observed_value' if 'observed_value' in bt_columns else None
    if value_field is None or not {'indicator_id', 'timestamp'}.issubset(bt_columns):
        return pd.DataFrame(columns=['source_ts', 'effective_ts', 'raw_value'])
    history_start = feature_ts - pd.Timedelta(hours=int(history_hours))
    feature_ts_sql = sql_timestamp(feature_ts)
    history_start_sql = sql_timestamp(history_start)
    release_sql = f"""
        SELECT timestamp AS source_ts,
               date_trunc('hour', timestamp + INTERVAL 2 HOUR) AS effective_ts,
               {value_field} AS raw_value
        FROM {BT_MACRO_TABLE}
        WHERE indicator_id = '{indicator_id}'
          AND {value_field} IS NOT NULL
          AND date_trunc('hour', timestamp + INTERVAL 2 HOUR) <= TIMESTAMP('{feature_ts_sql}')
          AND date_trunc('hour', timestamp + INTERVAL 2 HOUR) >= TIMESTAMP('{history_start_sql}')
        ORDER BY effective_ts ASC, source_ts ASC
    """
    anchor_sql = f"""
        SELECT timestamp AS source_ts,
               date_trunc('hour', timestamp + INTERVAL 2 HOUR) AS effective_ts,
               {value_field} AS raw_value
        FROM {BT_MACRO_TABLE}
        WHERE indicator_id = '{indicator_id}'
          AND {value_field} IS NOT NULL
          AND date_trunc('hour', timestamp + INTERVAL 2 HOUR) < TIMESTAMP('{history_start_sql}')
        ORDER BY effective_ts DESC, source_ts DESC
        LIMIT 1
    """
    release_pdf = query_to_pandas(release_sql, empty_columns=['source_ts', 'effective_ts', 'raw_value'])
    anchor_pdf = query_to_pandas(anchor_sql, empty_columns=['source_ts', 'effective_ts', 'raw_value'])
    history_pdf = pd.concat([anchor_pdf, release_pdf], ignore_index=True)
    if history_pdf.empty:
        return history_pdf
    history_pdf['source_ts'] = pd.to_datetime(history_pdf['source_ts'], errors='coerce')
    history_pdf['effective_ts'] = pd.to_datetime(history_pdf['effective_ts'], errors='coerce')
    history_pdf['raw_value'] = pd.to_numeric(history_pdf['raw_value'], errors='coerce')
    history_pdf = history_pdf.dropna(subset=['effective_ts', 'raw_value']).sort_values(['effective_ts', 'source_ts'])
    history_pdf = history_pdf.drop_duplicates(subset=['effective_ts'], keep='last')
    return history_pdf.reset_index(drop=True)


def build_macro_hourly_history(indicator_id, feature_ts, history_hours, include_current_hour=False):
    feature_ts = pd.to_datetime(feature_ts, errors='coerce')
    if pd.isna(feature_ts):
        return []
    release_history = load_macro_release_history(indicator_id, feature_ts, history_hours)
    if release_history.empty:
        return []
    end_ts = feature_ts if include_current_hour else feature_ts - pd.Timedelta(hours=1)
    start_ts = feature_ts - pd.Timedelta(hours=int(history_hours))
    if pd.isna(end_ts) or end_ts < start_ts:
        return []
    hourly_index = pd.date_range(start=start_ts, end=end_ts, freq='h')
    hourly_frame = pd.DataFrame({'timestamp': hourly_index})
    release_history = release_history[['effective_ts', 'raw_value']].rename(columns={'effective_ts': 'timestamp'})
    aligned = pd.merge_asof(hourly_frame, release_history.sort_values('timestamp'), on='timestamp', direction='backward')
    aligned['raw_value'] = pd.to_numeric(aligned['raw_value'], errors='coerce').ffill()
    aligned = aligned.dropna(subset=['raw_value'])
    return aligned['raw_value'].astype(float).tolist()


def compute_zscore_from_values(values):
    cleaned = [float(v) for v in values if pd.notna(v)]
    if not cleaned:
        return np.nan
    series = pd.Series(cleaned, dtype='float64')
    current_value = float(series.iloc[-1])
    mean_val = float(series.mean())
    std_val = float(series.std(ddof=1)) if len(series) > 1 else 0.0
    if math.isnan(std_val) or std_val == 0.0:
        return 0.0
    return (current_value - mean_val) / std_val


def build_feature_result(feature, label, model_input_value, source_value, source_ts, source_table, source_feature, reconstructed_value, freshness, freshness_status, note, lineage_source='N/A', lineage_reference_ts=None, lineage_reference_value=None, live_override_ts=None, live_override_value=None, upstream_source_table=None, upstream_source_field=None, upstream_source_ts=None, upstream_source_value=None, first_divergence='N/A', resolution_status='N/A'):
    diff, proof_status = evaluate_proof(feature, model_input_value, reconstructed_value)
    return {
        'feature': feature,
        'Feature': label,
        'Model input value': fmt_model_input_value(model_input_value),
        'Lineage reference table': _as_text(source_table),
        'Lineage reference feature': _as_text(source_feature),
        'Lineage reference timestamp': fmt_timestamp(source_ts),
        'Lineage reference value': fmt_model_input_value(source_value),
        'Upstream raw source table': _as_text(upstream_source_table),
        'Upstream raw source field': _as_text(upstream_source_field),
        'Upstream raw source timestamp': fmt_timestamp(upstream_source_ts),
        'Upstream raw source value': fmt_raw_snapshot_value(upstream_source_value),
        'Reconstructed value': fmt_model_input_value(reconstructed_value),
        'Validation diff': fmt_validation_diff(diff),
        'Proof status': proof_status,
        'Freshness': freshness,
        'Freshness status': freshness_status,
        'Lineage source type': lineage_source,
        'Live override timestamp': fmt_timestamp(live_override_ts),
        'Live override value': fmt_raw_snapshot_value(live_override_value),
        'Lineage reference timestamp/row': fmt_timestamp(lineage_reference_ts),
        'Lineage reference value/row': fmt_model_input_value(lineage_reference_value),
        'First point of divergence': _as_text(first_divergence),
        'Resolution status': _as_text(resolution_status),
        'Note': note,
    }


def unresolved_feature_result(feature, model_input_value, source_table, source_feature, source_field='N/A', note='Derived / unresolved', lineage_source='N/A', lineage_reference_ts=None, lineage_reference_value=None, live_override_ts=None, live_override_value=None, upstream_source_table='N/A', upstream_source_field='N/A', upstream_source_ts=None, upstream_source_value=None, first_divergence='N/A', resolution_status='N/A'):
    return {
        'feature': feature,
        'Feature': FEATURE_LABELS.get(feature, feature),
        'Model input value': fmt_model_input_value(model_input_value),
        'Lineage reference table': _as_text(source_table),
        'Lineage reference feature': _as_text(source_feature),
        'Lineage reference timestamp': 'N/A',
        'Lineage reference value': 'N/A',
        'Upstream raw source table': _as_text(upstream_source_table),
        'Upstream raw source field': _as_text(upstream_source_field),
        'Upstream raw source timestamp': fmt_timestamp(upstream_source_ts),
        'Upstream raw source value': fmt_raw_snapshot_value(upstream_source_value),
        'Reconstructed value': 'N/A',
        'Validation diff': 'N/A',
        'Proof status': 'Derived / unresolved',
        'Freshness': 'N/A',
        'Freshness status': 'Unresolved',
        'Lineage source type': lineage_source,
        'Live override timestamp': fmt_timestamp(live_override_ts),
        'Live override value': fmt_raw_snapshot_value(live_override_value),
        'Lineage reference timestamp/row': fmt_timestamp(lineage_reference_ts),
        'Lineage reference value/row': fmt_model_input_value(lineage_reference_value),
        'First point of divergence': _as_text(first_divergence),
        'Resolution status': _as_text(resolution_status),
        'Note': note,
    }


def validate_macro_feature(feature, model_input_value, feature_ts):
    spec = RAW_FEATURE_SPECS.get(feature, {})
    indicator_id = spec.get('indicator_id')
    if not indicator_id:
        return unresolved_feature_result(feature, model_input_value, SILVER_HIST_TABLE, feature, 'source not mapped', 'Missing macro spec mapping.')

    silver_bt_row = load_silver_bt_macro_row(feature_ts, feature)
    if silver_bt_row.empty:
        return unresolved_feature_result(feature, model_input_value, SILVER_HIST_TABLE, feature, 'carry-forward row', 'No carry-forward silver_bt row loaded for current model timestamp.')

    silver_bt_ts = silver_bt_row.get('timestamp')
    silver_bt_z = _to_float(silver_bt_row.get('z_value'))
    upstream_release = load_macro_release_asof(indicator_id, silver_bt_ts)

    source_type = 'silver_bt_master_grid_1h carry-forward'
    source_ts = silver_bt_ts
    source_feature = feature
    freshness, _ = fmt_freshness(source_ts, feature_ts)

    if pd.isna(silver_bt_z):
        return unresolved_feature_result(
            feature,
            model_input_value,
            SILVER_HIST_TABLE,
            source_feature,
            'z_value',
            'silver_bt carry-forward row has null z-score.',
            lineage_source=source_type,
            lineage_reference_ts=silver_bt_ts,
            lineage_reference_value=np.nan,
            upstream_source_table=BT_MACRO_TABLE,
            upstream_source_field='actual / observed_value',
            upstream_source_ts=upstream_release.get('source_ts'),
            upstream_source_value=upstream_release.get('raw_value'),
            first_divergence='silver_bt row returned null z-score',
            resolution_status='Unresolved before reconstruction',
        )

    model_diff_vs_bt = abs(_to_float(model_input_value) - silver_bt_z)

    live_row = load_live_macro_source(indicator_id, feature_ts) if spec.get('live_override') else pd.Series(dtype='object')
    if not live_row.empty:
        live_value = _to_float(live_row.get('raw_value'))
        if not pd.isna(live_value):
            history_values = build_macro_hourly_history(indicator_id, feature_ts, BATCH_Z_HISTORY_ROWS, include_current_hour=False)
            if history_values:
                live_recon = compute_zscore_from_values(history_values + [live_value])
                _, live_status = evaluate_proof(feature, model_input_value, live_recon)
                if live_status in {'Exact match to current model row', 'Close match to current model row'}:
                    source_type = 'RT live override used'
                    source_ts = live_row.get('source_ts')
                    freshness, _ = fmt_freshness(source_ts, feature_ts)
                    return build_feature_result(
                        feature,
                        FEATURE_LABELS.get(feature, feature),
                        model_input_value,
                        live_value,
                        source_ts,
                        live_row.get('source_table'),
                        live_row.get('source_field'),
                        live_recon,
                        freshness,
                        classify_market_freshness(source_ts, feature_ts),
                        'Live override value validated against model row and accepted as source.',
                        lineage_source=source_type,
                        lineage_reference_ts=silver_bt_ts,
                        lineage_reference_value=silver_bt_z,
                        live_override_ts=source_ts,
                        live_override_value=live_value,
                        upstream_source_table=BT_MACRO_TABLE,
                        upstream_source_field='actual / observed_value',
                        upstream_source_ts=upstream_release.get('source_ts'),
                        upstream_source_value=upstream_release.get('raw_value'),
                        first_divergence='None (proven live override match)',
                        resolution_status='Proven via live override reconstruction against raw macro release path',
                    )

    first_divergence = 'None (proven carry-forward match)' if model_diff_vs_bt <= Z_CLOSE_TOL else 'silver_rt_1h value differs from carry-forward silver_bt row'
    resolution_status = 'Proven via silver_bt lineage' if model_diff_vs_bt <= Z_CLOSE_TOL else 'Mismatch before model-input lineage boundary'

    return build_feature_result(
        feature,
        FEATURE_LABELS.get(feature, feature),
        model_input_value,
        silver_bt_z,
        source_ts,
        SILVER_HIST_TABLE,
        source_feature,
        silver_bt_z,
        freshness,
        'Fresh' if model_diff_vs_bt <= Z_CLOSE_TOL else 'Unresolved',
        'Prefer SILVER_BT row first (production lineage). Upstream raw check is diagnostic.',
        lineage_source=source_type,
        lineage_reference_ts=silver_bt_ts,
        lineage_reference_value=silver_bt_z,
        live_override_ts=pd.NaT,
        live_override_value=np.nan,
        upstream_source_table=BT_MACRO_TABLE,
        upstream_source_field='actual / observed_value',
        upstream_source_ts=upstream_release.get('source_ts'),
        upstream_source_value=upstream_release.get('raw_value'),
        first_divergence=first_divergence,
        resolution_status=resolution_status,
    )
def validate_rt_vix_feature(feature, model_input_value, feature_ts):
    spec = RAW_FEATURE_SPECS.get(feature, {})
    source_row = load_rt_vix_source(spec.get('series'), feature_ts)
    history_table = BT_VIX_TABLE if feature == 'VIXCLS_z' else BT_VIX3M_TABLE
    history_field = 'close' if feature == 'VIXCLS_z' else resolve_vix_history_value_column(history_table)
    if source_row.empty or not history_field:
        return unresolved_feature_result(feature, model_input_value, RT_VIX_TABLE, feature, spec.get('source_field', 'close'), 'RT VIX source or matching batch history path is unavailable.')
    history_values = load_history_values(history_table, history_field, feature_ts, LIVE_Z_HISTORY_ROWS)
    if not history_values:
        return unresolved_feature_result(feature, model_input_value, history_table, feature, history_field, 'Insufficient VIX history to reconstruct RT z-score.')
    reconstructed_value = compute_zscore_from_values(history_values + [float(source_row.get('raw_value'))])
    freshness, _ = fmt_freshness(source_row.get('source_ts'), feature_ts)
    note = 'Current raw input comes from bronze_rt_vix_1d; z-score history matches the RT enrichment batch window.'
    return build_feature_result(feature, FEATURE_LABELS.get(feature, feature), model_input_value, source_row.get('raw_value'), source_row.get('source_ts'), source_row.get('source_table'), source_row.get('source_field'), reconstructed_value, freshness, classify_market_freshness(source_row.get('source_ts'), feature_ts), note)


def load_market_history(feature_ts, lookback_hours):
    feature_ts = pd.to_datetime(feature_ts, errors='coerce')
    if pd.isna(feature_ts) or not table_exists(BT_MARKET_TABLE) or not table_exists(RT_MARKET_TABLE):
        return pd.Series(dtype='float64')
    history_start = feature_ts - pd.Timedelta(hours=lookback_hours)
    history_start_sql = sql_timestamp(history_start)
    feature_ts_sql = sql_timestamp(feature_ts)
    union_sql = f"""
        SELECT timestamp, close, source_rank
        FROM (
            SELECT timestamp, close, 0 AS source_rank
            FROM {BT_MARKET_TABLE}
            WHERE symbol = '{PRIMARY_SYMBOL}'
              AND timestamp >= TIMESTAMP('{history_start_sql}')
              AND timestamp <= TIMESTAMP('{feature_ts_sql}')
            UNION ALL
            SELECT timestamp, close, 1 AS source_rank
            FROM {RT_MARKET_TABLE}
            WHERE symbol = '{PRIMARY_SYMBOL}'
              AND timestamp >= TIMESTAMP('{history_start_sql}')
              AND timestamp <= TIMESTAMP('{feature_ts_sql}')
        ) t
        ORDER BY timestamp ASC, source_rank ASC
    """
    union_pdf = query_to_pandas(union_sql, empty_columns=['timestamp', 'close', 'source_rank'])
    anchor_sql = f"""
        SELECT timestamp, close, source_rank
        FROM (
            SELECT timestamp, close, 0 AS source_rank
            FROM {BT_MARKET_TABLE}
            WHERE symbol = '{PRIMARY_SYMBOL}'
              AND timestamp < TIMESTAMP('{history_start_sql}')
            UNION ALL
            SELECT timestamp, close, 1 AS source_rank
            FROM {RT_MARKET_TABLE}
            WHERE symbol = '{PRIMARY_SYMBOL}'
              AND timestamp < TIMESTAMP('{history_start_sql}')
        ) t
        ORDER BY timestamp DESC, source_rank DESC
        LIMIT 1
    """
    anchor_pdf = query_to_pandas(anchor_sql, empty_columns=['timestamp', 'close', 'source_rank'])
    if union_pdf.empty and anchor_pdf.empty:
        return pd.Series(dtype='float64')
    if not anchor_pdf.empty:
        anchor_row = anchor_pdf.iloc[0]
        anchor_add = pd.DataFrame([{'timestamp': pd.Timestamp(history_start), 'close': anchor_row.get('close'), 'source_rank': anchor_row.get('source_rank')}])
        union_pdf = pd.concat([anchor_add, union_pdf], ignore_index=True)
    union_pdf['timestamp'] = pd.to_datetime(union_pdf['timestamp'], errors='coerce')
    union_pdf['close'] = pd.to_numeric(union_pdf['close'], errors='coerce')
    union_pdf = union_pdf.dropna(subset=['timestamp', 'close']).sort_values(['timestamp', 'source_rank'])
    union_pdf = union_pdf.drop_duplicates(subset=['timestamp'], keep='last')
    hourly_index = pd.date_range(start=history_start, end=feature_ts, freq='h')
    market_series = union_pdf.set_index('timestamp')['close'].astype(float).reindex(hourly_index).ffill()
    if market_series.isna().any():
        market_series = market_series.bfill()
    return market_series


def validate_price_momentum_feature(model_input_value, feature_ts):
    source_row = load_rt_market_source(feature_ts)
    market_series = load_market_history(feature_ts, PRICE_CONTEXT_HOURS)
    feature_ts = pd.to_datetime(feature_ts, errors='coerce')
    lag_ts = feature_ts - pd.Timedelta(hours=24) if pd.notna(feature_ts) else pd.NaT
    if source_row.empty or market_series.empty or pd.isna(feature_ts) or lag_ts not in market_series.index:
        return unresolved_feature_result('Price_Mom_24H', model_input_value, RT_MARKET_TABLE, 'Price_Mom_24H', 'close', 'Unable to reconstruct RT market history path for 24H momentum.')
    lag_close = market_series.loc[lag_ts]
    current_close = _to_float(source_row.get('raw_value'))
    if pd.isna(lag_close) or pd.isna(current_close) or float(lag_close) == 0.0:
        return unresolved_feature_result('Price_Mom_24H', model_input_value, RT_MARKET_TABLE, 'Price_Mom_24H', 'close', 'Momentum reconstruction missing current or 24H lag close.')
    reconstructed_value = current_close / float(lag_close) - 1.0
    freshness, _ = fmt_freshness(source_row.get('source_ts'), feature_ts)
    note = 'Recomputed from the same RT market close plus 24H lag context used by RT enrichment.'
    return build_feature_result('Price_Mom_24H', FEATURE_LABELS.get('Price_Mom_24H', 'Price_Mom_24H'), model_input_value, current_close, source_row.get('source_ts'), source_row.get('source_table'), source_row.get('source_field'), reconstructed_value, freshness, classify_market_freshness(source_row.get('source_ts'), feature_ts), note)


def validate_vix_inversion_feature(model_input_value, feature_ts):
    vix_result = validate_rt_vix_feature('VIXCLS_z', np.nan, feature_ts)
    vix3m_result = validate_rt_vix_feature('vix3m_z', np.nan, feature_ts)
    vix_recon = _to_float(vix_result.get('Reconstructed value'))
    vix3m_recon = _to_float(vix3m_result.get('Reconstructed value'))
    if pd.isna(vix_recon) or pd.isna(vix3m_recon):
        return unresolved_feature_result('VIX_Inversion_Z', model_input_value, RT_VIX_TABLE, 'VIX_Inversion_Z', 'close (VIXCLS, vix3m)', 'Unable to reconstruct both RT VIX component z-scores.')
    source_row_vix = load_rt_vix_source('VIXCLS', feature_ts)
    source_row_vix3m = load_rt_vix_source('vix3m', feature_ts)
    source_ts = max(pd.to_datetime(source_row_vix.get('source_ts'), errors='coerce'), pd.to_datetime(source_row_vix3m.get('source_ts'), errors='coerce'))
    freshness, _ = fmt_freshness(source_ts, feature_ts)
    reconstructed_value = vix_recon - vix3m_recon
    raw_pair = 'N/A'
    if not source_row_vix.empty and not source_row_vix3m.empty:
        raw_pair = f"VIX {fmt_raw_snapshot_value(source_row_vix.get('raw_value'))} / VIX3M {fmt_raw_snapshot_value(source_row_vix3m.get('raw_value'))}"
    note = 'Derived from reconstructed RT VIX spot and VIX 3M z-scores.'
    return build_feature_result('VIX_Inversion_Z', FEATURE_LABELS.get('VIX_Inversion_Z', 'VIX_Inversion_Z'), model_input_value, raw_pair, source_ts, RT_VIX_TABLE, 'close (VIXCLS, vix3m)', reconstructed_value, freshness, classify_market_freshness(source_ts, feature_ts), note)


def validate_event_feature(feature, model_input_value, feature_ts, raw_column, indicator_id):
    note = 'Derived event-pulse flag not proven here; dashboard keeps it unresolved unless directly reconstructible.'
    return unresolved_feature_result(
        feature,
        model_input_value,
        SILVER_HIST_TABLE,
        feature,
        'N/A',
        note,
        lineage_source='Derived flag feature',
        resolution_status='Unresolved (event flag reconstruction skipped)',
    )

def validate_feature_against_model_row(feature, model_input_value, feature_ts):
    spec = RAW_FEATURE_SPECS.get(feature, {})
    feature_type = spec.get('type')
    if feature_type == 'macro':
        return validate_macro_feature(feature, model_input_value, feature_ts)
    if feature_type == 'rt_vix':
        return validate_rt_vix_feature(feature, model_input_value, feature_ts)
    if feature == 'VIX_Inversion_Z':
        return validate_vix_inversion_feature(model_input_value, feature_ts)
    if feature == 'Price_Mom_24H':
        return validate_price_momentum_feature(model_input_value, feature_ts)
    if feature == 'CPI_Event_Active':
        return validate_event_feature(feature, model_input_value, feature_ts, 'CPIAUCSL', 'CPIAUCSL')
    if feature == 'NFP_Event_Active':
        return validate_event_feature(feature, model_input_value, feature_ts, 'PAYEMS', 'PAYEMS')
    return unresolved_feature_result(feature, model_input_value, spec.get('source_table', 'N/A'), feature, spec.get('source_field', 'N/A'), spec.get('note', 'Direct raw source not displayed'))


latest_feed = latest_row(FEED_TABLE, 'feed_ts')
latest_health = latest_row(HEALTH_TABLE, 'rollup_ts')
latest_perf = latest_row(PERF_TABLE, 'rollup_ts')
execution_mode = _as_text(latest_feed.get('execution_mode', ''), '') if hasattr(latest_feed, 'get') else ''
if not execution_mode:
    execution_mode = _as_text(latest_perf.get('execution_mode', ''), '')
safe_execution_mode = execution_mode.replace("'", "''") if execution_mode else ''
mode_filter = '' if not safe_execution_mode else f"WHERE execution_mode = '{safe_execution_mode}'"

perf_history = query_to_pandas(f"""
    SELECT *
    FROM {PERF_TABLE}
    {mode_filter}
    ORDER BY rollup_ts DESC
    LIMIT {LOOKBACK_HOURS}
""")

health_history = query_to_pandas(f"""
    SELECT *
    FROM {HEALTH_TABLE}
    {mode_filter}
    ORDER BY rollup_ts DESC
    LIMIT {LOOKBACK_HOURS}
""")

signal_history = query_to_pandas(f"""
    SELECT *
    FROM (
      SELECT timestamp, regime_name, CAST(regime_int AS INT) AS regime_int, position
      FROM {REGIME_SIGNAL_TABLE}
      ORDER BY timestamp DESC
      LIMIT {LOOKBACK_HOURS}
    ) t
    ORDER BY timestamp ASC
""")

silver_cols = table_columns(SILVER_RT_TABLE)
if silver_cols:
    requested = [f for f in FEATURES if f in silver_cols]
    query_cols = ', '.join(['timestamp'] + requested)
    feature_history = query_to_pandas(f"SELECT {query_cols} FROM {SILVER_RT_TABLE} ORDER BY timestamp DESC LIMIT {LOOKBACK_HOURS}")
else:
    feature_history = pd.DataFrame(columns=['timestamp'] + FEATURES)
if 'timestamp' in feature_history.columns:
    feature_history['timestamp'] = pd.to_datetime(feature_history['timestamp'], errors='coerce')
    feature_history = feature_history.sort_values('timestamp').reset_index(drop=True)
    for col in [c for c in feature_history.columns if c != 'timestamp']:
        feature_history[col] = pd.to_numeric(feature_history[col], errors='coerce')

trade_history = query_to_pandas(f"SELECT * FROM {TRADE_LOG_TABLE} ORDER BY trade_ts DESC LIMIT 250", empty_columns=['trade_ts'])
if 'trade_ts' in trade_history.columns:
    trade_history['trade_ts'] = pd.to_datetime(trade_history['trade_ts'], errors='coerce')
    trade_history = trade_history.sort_values('trade_ts').reset_index(drop=True)

model, model_version, model_path = load_model()
importance_frame = feature_importance(model)
importance_value_map = dict(zip(importance_frame['feature'], importance_frame['importance']))
importance_pct_map = dict(zip(importance_frame['feature'], importance_frame['importance_pct']))
probability_history = pd.DataFrame(columns=['timestamp'] + list(PROBABILITY_LABELS.keys()))
if not feature_history.empty and 'timestamp' in feature_history.columns:
    probability_history = pd.concat(
        [
            feature_history[['timestamp']].reset_index(drop=True),
            compute_probabilities(feature_history, model).reset_index(drop=True),
        ],
        axis=1,
    )

perf_history = perf_history.sort_values('rollup_ts').reset_index(drop=True) if not perf_history.empty else perf_history
if not perf_history.empty:
    perf_history['rollup_ts'] = pd.to_datetime(perf_history['rollup_ts'], errors='coerce')
    for c in ['account_equity', 'account_cash', 'drawdown_pct', 'regime_drawdown_pct', 'alpha_pct', 'strategy_return_1h', 'alpha_1h', 'strategy_return_1d', 'alpha_1d']:
        if c in perf_history.columns:
            perf_history[c] = pd.to_numeric(perf_history[c], errors='coerce')

if not health_history.empty:
    health_history['rollup_ts'] = pd.to_datetime(health_history['rollup_ts'], errors='coerce')
    health_history = health_history.sort_values('rollup_ts').reset_index(drop=True)

if not signal_history.empty:
    signal_history['timestamp'] = pd.to_datetime(signal_history['timestamp'], errors='coerce')
    signal_history = signal_history.sort_values('timestamp').reset_index(drop=True)
    signal_history['regime_int'] = pd.to_numeric(signal_history['regime_int'], errors='coerce')
    signal_history['regime_name'] = [regime_label(name, code) for name, code in zip(signal_history.get('regime_name', pd.Series(index=signal_history.index, dtype='object')), signal_history.get('regime_int', pd.Series(index=signal_history.index, dtype='float64')))]

regime_history = signal_history.copy()
regime_intervals = build_regime_intervals(signal_history)
latest_probability_rows = probability_history.dropna(subset=list(PROBABILITY_LABELS.keys()), how='all') if not probability_history.empty else pd.DataFrame()
latest_probability = latest_probability_rows.iloc[-1] if not latest_probability_rows.empty else pd.Series(dtype='float64')


displayHTML('<h2>1. Executive decision header</h2>')
prob_label, prob_value = top_named_value_from_row(latest_probability, PROBABILITY_LABELS)
kpi = [
    ('Current regime', _as_text(latest_feed.get('regime_name', 'N/A'))),
    ('Current model probability', f'{prob_label}: {fmt_pct(prob_value)}' if pd.notna(_to_float(prob_value)) else 'N/A'),
    ('Historical regime mix', historical_mix_text(latest_feed)),
    ('Current position', _as_text(latest_feed.get('actual_ticker', latest_feed.get('position', 'N/A')))),
    ('Execution mode', _as_text(latest_feed.get('execution_mode', latest_perf.get('execution_mode', 'N/A')))),
    ('Account equity', fmt_money(latest_feed.get('account_equity'))),
    ('Account cash', fmt_money(latest_feed.get('account_cash'))),
    ('Account balance', fmt_money(latest_feed.get('account_balance'))),
    ('1h strategy', fmt_pct(latest_perf.get('strategy_return_1h'))),
    ('1h alpha', fmt_pct(latest_perf.get('alpha_1h'))),
    ('1d strategy', fmt_pct(latest_perf.get('strategy_return_1d'))),
    ('1d alpha', fmt_pct(latest_perf.get('alpha_1d'))),
    ('Current drawdown', fmt_pct(latest_feed.get('drawdown_pct'))),
    ('Freshness (min)', fmt_minutes(latest_feed.get('dashboard_data_freshness_minutes'))),
    ('Latest audit timestamp', fmt_timestamp(latest_feed.get('latest_audit_ts'))),
    ('Last execution action', _as_text(latest_feed.get('last_execution_action'))),
    ('Skip reason', _as_text(latest_feed.get('last_skip_reason'))),
    ('Error reason', _as_text(latest_feed.get('last_error_reason'))),
]
display(pd.DataFrame(kpi, columns=['Metric', 'Value']))


displayHTML('<h2>2. Strategy vs SPY performance</h2>')
if perf_history.empty:
    print('No strategy performance history is available yet.')
else:
    aligned = perf_history.copy()
    aligned = aligned.dropna(subset=['rollup_ts', 'account_equity'])
    if aligned.empty:
        print('No aligned performance records available.')
    else:
        aligned['strategy_return'] = aligned['account_equity'] / aligned['account_equity'].iloc[0] - 1.0
        if 'account_cash' in aligned.columns and aligned['account_cash'].notna().all():
            aligned['cash_return'] = aligned['account_cash'] / aligned['account_cash'].iloc[0] - 1.0

        spy_df = pd.DataFrame()
        if 'spy_close' in trade_history.columns:
            spy_df = trade_history[['trade_ts', 'spy_close']].copy() if 'trade_ts' in trade_history.columns else pd.DataFrame()
            spy_df = spy_df.dropna()

        if not spy_df.empty:
            spy_df = spy_df.sort_values('trade_ts')
            aligned = pd.merge_asof(
                aligned.sort_values('rollup_ts'),
                spy_df.sort_values('trade_ts'),
                left_on='rollup_ts',
                right_on='trade_ts',
                direction='backward'
            )
            aligned['spy_close'] = pd.to_numeric(aligned['spy_close'], errors='coerce')
            if aligned['spy_close'].notna().any():
                first_spy = float(aligned['spy_close'].dropna().iloc[0])
                aligned['spy_return'] = aligned['spy_close'] / first_spy - 1.0

        fig = plt.figure(figsize=(16, 10), constrained_layout=True)
        gs = fig.add_gridspec(3, 1, height_ratios=[3.2, 1.2, 1.0])
        ax_perf = fig.add_subplot(gs[0, 0])
        ax_alpha = fig.add_subplot(gs[1, 0], sharex=ax_perf)
        ax_dd = fig.add_subplot(gs[2, 0], sharex=ax_perf)

        if not regime_intervals.empty:
            window_start = aligned['rollup_ts'].iloc[0]
            window_end = aligned['rollup_ts'].iloc[-1]
            for idx, row in regime_intervals.iterrows():
                start_t = row['start_ts']
                end_t = row['end_ts'] if pd.notna(row.get('end_ts')) else window_end
                if pd.isna(start_t) or pd.isna(end_t):
                    continue
                if idx == 0 and start_t > window_start:
                    start_t = window_start
                if start_t < window_start:
                    start_t = window_start
                if start_t >= window_end or end_t <= start_t:
                    continue
                color = REGIME_COLOR.get(_as_text(row['regime_name'], 'Unknown'), '#9E9E9E')
                ax_perf.axvspan(start_t, min(end_t, window_end), color=color, alpha=0.15)

        ax_perf.plot(aligned['rollup_ts'], aligned['strategy_return'] * 100, linewidth=2.2, label='Strategy')
        if 'spy_return' in aligned.columns:
            ax_perf.plot(aligned['rollup_ts'], aligned['spy_return'] * 100, linewidth=1.5, linestyle='--', label='SPY proxy', alpha=0.8)
        ax_perf.set_title('Saxo strategy vs SPY')
        ax_perf.set_ylabel('Return (%)')
        ax_perf.grid(alpha=0.3)
        ax_perf.legend(loc='upper left')

        ax_alpha.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('alpha_pct', np.nan), errors='coerce') * 100, linewidth=2.0, label='Alpha')
        ax_alpha.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('alpha_1h', np.nan), errors='coerce') * 100, linewidth=1.3, linestyle=':', label='Alpha 1h')
        ax_alpha.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('alpha_1d', np.nan), errors='coerce') * 100, linewidth=1.3, linestyle='--', label='Alpha 1d')
        ax_alpha.axhline(0, color='black', linestyle='--', alpha=0.5)
        ax_alpha.set_ylabel('Alpha (%)')
        ax_alpha.set_title('Alpha panel')
        ax_alpha.grid(alpha=0.3)
        ax_alpha.legend(loc='upper left')

        drawdown_series = pd.to_numeric(aligned.get('drawdown_pct', np.nan), errors='coerce').fillna(0)
        ax_dd.plot(aligned['rollup_ts'], drawdown_series * 100, linewidth=1.8, color='#B03A2E', label='Strategy drawdown')
        if 'regime_drawdown_pct' in aligned.columns:
            ax_dd.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('regime_drawdown_pct', np.nan), errors='coerce') * 100, linewidth=1.3, color='#D35400', label='Regime drawdown')
        ax_dd.axhline(0, color='black', linestyle='--', alpha=0.5)
        ax_dd.set_ylabel('Drawdown (%)')
        ax_dd.set_title('Drawdown panel')
        ax_dd.grid(alpha=0.3)
        ax_dd.legend(loc='lower right')
        plt.show()


displayHTML('<h2>3. Historical regime / signal monitoring</h2>')
if not regime_history.empty:
    displayHTML('<p><i>Model probabilities below are computed on the fly from the current promoted model using recent feature history. Hourly regime history remains sourced from bronze_rt_regime_signal.</i></p>')
    regime_head_cols = [c for c in ['timestamp', 'regime_name', 'regime_int', 'position'] if c in regime_history.columns]
    regime_head = regime_history[regime_head_cols].copy()
    regime_head['timestamp'] = regime_head['timestamp'].apply(fmt_timestamp)
    regime_head = regime_head.sort_values('timestamp', ascending=False).head(24)
    regime_head = regime_head.rename(columns={'timestamp': 'Timestamp', 'regime_name': 'Regime', 'regime_int': 'Regime code', 'position': 'Target position'})
    display(regime_head)

    fig2, (axp, axr) = plt.subplots(2, 1, figsize=(16, 7.2), sharex=True, gridspec_kw={'height_ratios': [2.1, 1.0]}, constrained_layout=True)
    probability_plot_columns = [column for column in PROBABILITY_LABELS.keys() if column in probability_history.columns and probability_history[column].notna().any()]
    if probability_plot_columns:
        for column in probability_plot_columns:
            axp.step(probability_history['timestamp'], probability_history[column] * 100, where='post', linewidth=1.7, label=PROBABILITY_LABELS[column])
        axp.set_title('Historical model probabilities (computed on the fly)')
        axp.set_ylabel('Probability (%)')
        axp.set_ylim(0, 100)
        axp.grid(alpha=0.25)
        axp.legend(loc='upper left', ncol=3)
    else:
        axp.axis('off')
        axp.text(0.5, 0.5, 'Model probability history is unavailable because predict_proba could not be computed.', ha='center', va='center')

    sig_plot = regime_history[['timestamp', 'regime_int']].dropna().copy()
    if not sig_plot.empty:
        sig_plot['regime_step'] = pd.to_numeric(sig_plot['regime_int'], errors='coerce')
        axr.step(sig_plot['timestamp'], sig_plot['regime_step'], where='post', linewidth=2.0, color='#1F4E79')
        axr.set_ylim(-0.1, 2.1)
        axr.set_ylabel('Regime')
        axr.set_title('Hourly regime history')
        axr.set_yticks([0, 1, 2])
        axr.set_yticklabels(['Expansion', 'Slowdown', 'Recession'])
        axr.grid(alpha=0.25)
    else:
        axr.axis('off')
        axr.text(0.5, 0.5, 'No signal history available.', ha='center', va='center')

    plt.show()
else:
    print('No regime signal history available from bronze_rt_regime_signal.')


displayHTML('<h2>4. Current feature state</h2>')
current_features = pd.DataFrame(columns=['feature', 'label', 'model_importance', 'importance_pct', 'current_z_score', 'delta_vs_prior', 'weighted_move_proxy'])
if feature_history.empty:
    print('No feature history available in silver.')
else:
    latest = feature_history.tail(1).iloc[0]
    prev = feature_history.tail(2).iloc[0] if len(feature_history) > 1 else pd.Series(dtype=float)
    rows = []
    for feature in FEATURES:
        if feature not in feature_history.columns:
            continue
        cur = _to_float(latest.get(feature))
        p = _to_float(prev.get(feature)) if not prev.empty else np.nan
        delta_z = cur - p if not pd.isna(cur) and not pd.isna(p) else np.nan
        model_importance = _to_float(importance_value_map.get(feature, np.nan))
        importance_pct = _to_float(importance_pct_map.get(feature, np.nan))
        rows.append((
            feature,
            FEATURE_LABELS.get(feature, feature),
            model_importance,
            importance_pct,
            _to_float(cur),
            delta_z,
            abs(importance_pct * delta_z) if not pd.isna(delta_z) and not pd.isna(importance_pct) else np.nan,
        ))

    current_features = pd.DataFrame(rows, columns=['feature', 'label', 'model_importance', 'importance_pct', 'current_z_score', 'delta_vs_prior', 'weighted_move_proxy'])
    current_features = current_features.sort_values('model_importance', ascending=False).reset_index(drop=True)
    displayHTML('<p><i>First view ranks the largest recent weighted movers; second view ranks the highest model-importance features.</i></p>')
    feature_display_columns = {'label': 'Feature', 'model_importance': 'Model importance', 'importance_pct': 'Importance %', 'current_z_score': 'Current z-score', 'delta_vs_prior': 'Delta vs previous bar', 'weighted_move_proxy': 'Weighted move proxy'}
    weighted_movers_display = current_features.sort_values('weighted_move_proxy', ascending=False).head(TOP_FEATURES).drop(columns=['feature']).rename(columns=feature_display_columns)
    importance_leaders_display = current_features.sort_values('model_importance', ascending=False).head(TOP_FEATURES).drop(columns=['feature']).rename(columns=feature_display_columns)
    displayHTML('<h4>Largest recent weighted movers</h4>')
    display(weighted_movers_display.style.format({'Model importance': '{:.4f}', 'Importance %': '{:.2%}', 'Current z-score': '{:+.3f}', 'Delta vs previous bar': '{:+.3f}', 'Weighted move proxy': '{:.4f}'}, na_rep='N/A'))
    displayHTML('<h4>Highest model importance</h4>')
    display(importance_leaders_display.style.format({'Model importance': '{:.4f}', 'Importance %': '{:.2%}', 'Current z-score': '{:+.3f}', 'Delta vs previous bar': '{:+.3f}', 'Weighted move proxy': '{:.4f}'}, na_rep='N/A'))
    displayHTML('<h4>Current raw feature snapshot</h4>')
    displayHTML('<p><i>Lineage-aware validation: first validates silver_rt_1h against silver_bt_master_grid_1h feature lineage; unresolved/override states are then traced to raw macro events for diagnostics only.</i></p>')
    latest_feature_ts = latest.get('timestamp') if 'timestamp' in latest.index else pd.NaT
    validation_rows = []
    for feature in [f for f in FEATURES if f in feature_history.columns]:
        validation_rows.append(validate_feature_against_model_row(feature, latest.get(feature), latest_feature_ts))
    validation_df = pd.DataFrame(validation_rows)

    exact_count = int((validation_df['Proof status'] == 'Exact match to current model row').sum()) if not validation_df.empty else 0
    close_count = int((validation_df['Proof status'] == 'Close match to current model row').sum()) if not validation_df.empty else 0
    mismatch_count = int((validation_df['Proof status'] == 'Mismatch vs current model row').sum()) if not validation_df.empty else 0
    unresolved_count = int((validation_df['Proof status'] == 'Derived / unresolved').sum()) if not validation_df.empty else 0
    blocker_features = validation_df.loc[validation_df['Proof status'].isin(['Mismatch vs current model row', 'Derived / unresolved']), 'Feature'].tolist() if not validation_df.empty else []
    blocker_text = ', '.join(blocker_features) if blocker_features else 'None'

    proof_summary = pd.DataFrame({
        'Metric': ['Total features checked', 'Exact matches', 'Close matches', 'Mismatches', 'Derived / unresolved', 'Critical blockers'],
        'Value': [str(len(validation_df)), str(exact_count), str(close_count), str(mismatch_count), str(unresolved_count), str(blocker_text)],
    })
    display(proof_summary)

    if mismatch_count > 0:
        displayHTML('<p style="color:#B03A2E;"><b>Do not treat current raw snapshot as fully validated; one or more features do not reproduce the current model row.</b></p>')

    raw_snapshot_columns = [
        'Feature',
        'Model input value',
        'Reconstructed value',
        'Validation diff',
        'Proof status',
        'Lineage reference table',
        'Lineage reference feature',
        'Lineage reference timestamp',
        'Lineage reference value',
        'Lineage reference timestamp/row',
        'Lineage reference value/row',
        'Upstream raw source table',
        'Upstream raw source field',
        'Upstream raw source timestamp',
        'Upstream raw source value',
        'Freshness',
        'Freshness status',
        'Lineage source type',
        'Live override timestamp',
        'Live override value',
        'First point of divergence',
        'Resolution status',
        'Note',
    ]
    if not validation_df.empty:
        displayHTML('<h5>Feature proof (top features by model importance)</h5>')
        top_snapshot_features = current_features[['feature', 'model_importance']].copy()
        raw_snapshot_df = validation_df.merge(top_snapshot_features, on='feature', how='left')
        raw_snapshot_df = raw_snapshot_df.sort_values('model_importance', ascending=False).head(TOP_FEATURES)
        display(raw_snapshot_df[raw_snapshot_columns])

        macro_feature_ids = {f for f in [f for f in RAW_FEATURE_SPECS.keys()] if RAW_FEATURE_SPECS[f].get('type') == 'macro'}
        macro_lineage_cols = [
            'feature',
            'Feature',
            'Lineage source type',
            'Lineage reference table',
            'Lineage reference feature',
            'Lineage reference timestamp',
            'Lineage reference value',
            'Lineage reference timestamp/row',
            'Lineage reference value/row',
            'Upstream raw source table',
            'Upstream raw source field',
            'Upstream raw source timestamp',
            'Upstream raw source value',
            'Live override timestamp',
            'Live override value',
            'First point of divergence',
            'Resolution status',
            'Proof status',
            'Note',
        ]
        macro_lineage_debug = validation_df[[c for c in macro_lineage_cols if c in validation_df.columns]].copy()
        macro_lineage_debug = macro_lineage_debug[macro_lineage_debug['feature'].isin(macro_feature_ids)]
        if not macro_lineage_debug.empty:
            displayHTML('<h5>Macro lineage debug (diagnostic)</h5>')
            display(macro_lineage_debug)






displayHTML('<h3>4b. Macro public validation with ALFRED/FRED (last 12 months)</h3>')
displayHTML('<p><i>Lineage proof validates that the model-input row is consistent with production feature lineage. This section validates that the upstream macro raw input is also consistent with ALFRED/FRED reference data.</i></p>')

if 'validation_df' not in globals() or validation_df.empty:
    print('Validation table is not available yet; run section 4 current feature state first.')
elif feature_history.empty:
    print('feature_history is empty; skipping public macro comparison.')
else:
    macro_input_rows = []
    for feature in ['CPIAUCSL_z', 'PCEPI_z', 'PCE_z', 'PAYEMS_z', 'UNRATE_z', 'GDPC1_z', 'INDPRO_z', 'UMCSENT_z']:
        row = validation_df[validation_df['feature'] == feature]
        if row.empty:
            continue
        row = row.iloc[0]
        macro_input_rows.append({
            'feature': FEATURE_LABELS.get(feature, feature),
            'series_id': RAW_FEATURE_SPECS.get(feature, {}).get('indicator_id', 'N/A'),
            'lineage_raw_timestamp': row.get('Upstream raw source timestamp'),
            'lineage_raw_value': row.get('Upstream raw source value'),
            'lineage_source_table': row.get('Upstream raw source table'),
            'lineage_source_field': row.get('Upstream raw source field'),
        })

    macro_input_df = pd.DataFrame(macro_input_rows)
    if macro_input_df.empty:
        print('No macro lineage inputs available for public validation.')
    else:
        macro_input_df['series_id'] = macro_input_df['series_id'].fillna('N/A')
        macro_input_df = macro_input_df[macro_input_df['series_id'].isin([s for s in ALFRED_SERIES_IDS])]
        displayHTML('<h5>A. Public macro validation input table</h5>')
        input_view = macro_input_df[[
            'feature',
            'series_id',
            'lineage_raw_timestamp',
            'lineage_raw_value',
            'lineage_source_table',
            'lineage_source_field',
        ]].copy()
        input_view['lineage_raw_timestamp'] = input_view['lineage_raw_timestamp'].apply(fmt_timestamp)
        display(input_view)

        lookup_frames = []
        for series_id in sorted(set(macro_input_df['series_id'].tolist())):
            series_public = fetch_alfred_public_lookup(series_id, lookback_days=ALFRED_SERIES_LOOKBACK_DAYS)
            if series_public.empty:
                continue
            lookup_frames.append(series_public[['series_id', 'observation_date', 'realtime_start', 'realtime_end', 'public_value', 'public_source']])
        public_lookup_df = pd.concat(lookup_frames, ignore_index=True) if lookup_frames else pd.DataFrame(columns=['series_id', 'observation_date', 'realtime_start', 'realtime_end', 'public_value', 'public_source'])
        displayHTML('<h5>B. ALFRED/FRED public lookup table</h5>')
        if public_lookup_df.empty:
            print('No ALFRED/FRED lookup data available for configured series in the recent lookback window.')
        else:
            public_view = public_lookup_df.copy()
            public_view['observation_date'] = public_view['observation_date'].dt.strftime('%Y-%m-%d')
            public_view['realtime_start'] = public_view['realtime_start'].dt.strftime('%Y-%m-%d')
            public_view['realtime_end'] = public_view['realtime_end'].dt.strftime('%Y-%m-%d')
            display(public_view.head(250))

        final_comparison = validate_alfred_public_rows(macro_input_df)
        if final_comparison.empty:
            print('No final comparison rows could be built.')
        else:
            displayHTML('<h5>C. Final lineage vs ALFRED/FRED comparison</h5>')
            comp_view = final_comparison[[
                'feature',
                'series_id',
                'lineage_raw_timestamp',
                'lineage_raw_value',
                'public_observation_date',
                'public_realtime_start',
                'public_value',
                'public_source',
                'comparison_result',
                'comparison_note',
            ]].copy()
            comp_view['lineage_raw_timestamp'] = comp_view['lineage_raw_timestamp'].apply(fmt_timestamp)
            comp_view['public_observation_date'] = comp_view['public_observation_date'].apply(fmt_timestamp)
            comp_view['public_realtime_start'] = comp_view['public_realtime_start'].apply(fmt_timestamp)
            display(comp_view)

            summary = []
            for series_id, group in final_comparison.groupby('series_id'):
                summary.append({
                    'series_id': series_id,
                    'rows_compared': int(group.shape[0]),
                    'exact_match_count': int((group['comparison_result'] == 'exact_match').sum()),
                    'pit_consistent_count': int((group['comparison_result'] == 'pit_consistent').sum()),
                    'value_mismatch_count': int((group['comparison_result'] == 'value_mismatch').sum()),
                    'timestamp_mismatch_count': int((group['comparison_result'] == 'timestamp_mismatch').sum()),
                    'unresolved_count': int((group['comparison_result'] == 'unresolved').sum()),
                })

            summary_df = pd.DataFrame(summary)
            if summary_df.empty:
                summary_df = pd.DataFrame([{
                    'series_id': 'N/A',
                    'rows_compared': 0,
                    'exact_match_count': 0,
                    'pit_consistent_count': 0,
                    'value_mismatch_count': 0,
                    'timestamp_mismatch_count': 0,
                    'unresolved_count': 0,
                }])
            displayHTML('<h5>D. Summary by series</h5>')
            display(summary_df)

            unresolved_count = int((final_comparison['comparison_result'] == 'unresolved').sum())
            mismatch_count = int((final_comparison['comparison_result'].isin(['value_mismatch', 'timestamp_mismatch'])).sum())
            if (unresolved_count + mismatch_count) > 0:
                displayHTML('<p style="color:#B03A2E;"><b>Public/lineage differences remain for one or more macro rows; review unresolved or timestamp/value mismatches.</b></p>')
            else:
                displayHTML('<p style="color:#2E7D32;"><b>No clear public-data mismatches found in the latest ALFRED/FRED overlap.</b></p>')

            diagnostic_export = final_comparison.copy()
            diagnostic_export['lineage_raw_timestamp'] = diagnostic_export['lineage_raw_timestamp'].apply(fmt_timestamp)
            diagnostic_export['public_observation_date'] = diagnostic_export['public_observation_date'].apply(fmt_timestamp)
            diagnostic_export['public_realtime_start'] = diagnostic_export['public_realtime_start'].apply(fmt_timestamp)
            diagnostic_export['public_realtime_end'] = diagnostic_export['public_realtime_end'].apply(fmt_timestamp)
            try:
                diagnostic_export.to_csv(ALFRED_DIAGNOSTIC_CSV_PATH, index=False)
                print(f'ALFRED/FRED diagnostic CSV written to: {ALFRED_DIAGNOSTIC_CSV_PATH}')
            except Exception:
                print(f'ALFRED/FRED diagnostic CSV write skipped/unavailable: {ALFRED_DIAGNOSTIC_CSV_PATH}')

displayHTML('<h2>5. Historical feature monitoring</h2>')

if feature_history.empty or current_features.empty:
    print('No feature monitoring data available.')
else:
    displayHTML('<p><i>Recent feature tracking combines the most decision-relevant series, while 24H momentum stays on a dedicated right axis for visibility.</i></p>')
    recent_feature_window = feature_history.tail(24 * FEATURE_PLOT_DAYS).copy()
    combined_features = [feature for feature in RECENT_TRACKING_FEATURES if feature in recent_feature_window.columns and feature != 'Price_Mom_24H']
    if not recent_feature_window.empty and combined_features:
        fig_combo, ax_combo = plt.subplots(figsize=(16, 5.2), constrained_layout=True)
        combo_colors = {'VIX_Inversion_Z': '#1F4E79', 'VIXCLS_z': '#B03A2E', 'vix3m_z': '#7D6608', 'PAYEMS_z': '#117A65', 'UNRATE_z': '#6C3483'}
        for feature in combined_features:
            ax_combo.plot(recent_feature_window['timestamp'], pd.to_numeric(recent_feature_window[feature], errors='coerce'), linewidth=1.5, label=FEATURE_LABELS.get(feature, feature), color=combo_colors.get(feature, '#1F4E79'))
        ax_combo.axhline(0, color='black', linestyle=':', alpha=0.45)
        ax_combo.set_title(f'Recent Feature Tracking ({FEATURE_PLOT_DAYS}D)')
        ax_combo.set_ylabel('Feature z-score')
        ax_combo.grid(alpha=0.25)
        momentum_axis = None
        if 'Price_Mom_24H' in recent_feature_window.columns:
            momentum_axis = ax_combo.twinx()
            momentum_axis.plot(recent_feature_window['timestamp'], pd.to_numeric(recent_feature_window['Price_Mom_24H'], errors='coerce'), linewidth=1.8, linestyle='--', color='#C0392B', label=FEATURE_LABELS.get('Price_Mom_24H', 'Price_Mom_24H'))
            momentum_axis.axhline(0, color='#C0392B', linestyle=':', alpha=0.3)
            momentum_axis.set_ylabel('24H momentum')
        combo_handles, combo_labels = ax_combo.get_legend_handles_labels()
        if momentum_axis is not None:
            momentum_handles, momentum_labels = momentum_axis.get_legend_handles_labels()
            combo_handles += momentum_handles
            combo_labels += momentum_labels
        ax_combo.legend(combo_handles, combo_labels, loc='upper left', ncol=3)
        plt.show()

    displayHTML('<p><i>Top feature history by model importance over the trailing window.</i></p>')
    panel_features = [f for f in current_features['feature'].tolist() if f in feature_history.columns and f != 'Price_Mom_24H'][:6]
    selected_window = recent_feature_window[['timestamp'] + panel_features].copy()
    if panel_features:
        n_cols = 2 if len(panel_features) > 1 else 1
        n_rows = math.ceil(len(panel_features) / n_cols)
        fig3, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.2 * n_rows), sharex=True, constrained_layout=True)
        axes = np.atleast_1d(axes).ravel()
        for axis, feature in zip(axes, panel_features):
            series = pd.to_numeric(selected_window[feature], errors='coerce')
            axis.plot(selected_window['timestamp'], series, linewidth=1.5, color='#1F4E79')
            axis.set_ylabel('z-score')
            axis.axhline(0, color='black', linestyle=':', alpha=0.5)
            axis.set_title(FEATURE_LABELS.get(feature, feature))
            axis.grid(alpha=0.3)
        for axis in axes[len(panel_features):]:
            axis.axis('off')
        plt.show()
    else:
        print('No top feature series available in silver_rt_1h.')

    if 'Price_Mom_24H' in feature_history.columns:
        mom = feature_history[['timestamp', 'Price_Mom_24H']].dropna()
        if not mom.empty:
            mom_window = mom.tail(24 * FEATURE_PLOT_DAYS)
            fig4, ax4 = plt.subplots(figsize=(16, 3.8), constrained_layout=True)
            ax4.bar(mom_window['timestamp'], mom_window['Price_Mom_24H'], color=['#2E8B57' if v >= 0 else '#C0392B' for v in mom_window['Price_Mom_24H']], width=0.03, alpha=0.65)
            ax4.set_title('24H momentum (dedicated panel)')
            ax4.set_ylabel('z-score')
            ax4.set_xlabel('Timestamp')
            ax4.axhline(0, color='black', linestyle='--', alpha=0.5)
            ax4.grid(alpha=0.3)
            ax4.margins(x=0.01)
            plt.show()


displayHTML('<h2>6. Attribution</h2>')
displayHTML('<p><i>Top weighted movers vs previous bar. Descriptive proxy only; not causal and not a SHAP explanation.</i></p>')
if feature_history.empty or current_features.empty:
    print('No attribution history available.')
else:
    probability_context_df = pd.DataFrame(
        [
            {
                'Top probability': f'{prob_label}: {fmt_pct(prob_value)}' if pd.notna(_to_float(prob_value)) else 'N/A',
                'Expansion': fmt_pct(latest_probability.get('prob_expansion')),
                'Slowdown': fmt_pct(latest_probability.get('prob_slowdown')),
                'Recession': fmt_pct(latest_probability.get('prob_recession')),
            }
        ]
    )
    display(probability_context_df)
    bridge_display = current_features.sort_values('weighted_move_proxy', ascending=False).head(5).drop(columns=['feature']).copy()
    bridge_display = bridge_display.rename(columns={'label': 'Feature', 'model_importance': 'Model importance', 'importance_pct': 'Importance %', 'current_z_score': 'Current z-score', 'delta_vs_prior': 'Delta vs previous bar', 'weighted_move_proxy': 'Weighted move proxy'})
    display(bridge_display.style.format({'Model importance': '{:.4f}', 'Importance %': '{:.2%}', 'Current z-score': '{:+.3f}', 'Delta vs previous bar': '{:+.3f}', 'Weighted move proxy': '{:.4f}'}, na_rep='N/A'))


displayHTML('<h2>7. Historical signals and trades</h2>')
sig_cols = [
    'timestamp', 'regime_name', 'regime_int', 'position'
]
if not signal_history.empty:
    available_sig = [c for c in sig_cols if c in signal_history.columns]
    print('Latest 24 hourly signals:')
    signal_tail = signal_history[available_sig].sort_values('timestamp', ascending=False).head(24).copy()
    if 'timestamp' in signal_tail.columns:
        signal_tail['timestamp'] = signal_tail['timestamp'].apply(fmt_timestamp)
    if 'regime_name' in signal_tail.columns:
        signal_tail['regime_name'] = signal_tail['regime_name'].apply(_as_text)
    if 'position' in signal_tail.columns:
        signal_tail['position'] = signal_tail['position'].apply(_as_text)
    signal_tail = signal_tail.rename(columns={'timestamp': 'Timestamp', 'regime_name': 'Regime', 'regime_int': 'Regime code', 'position': 'Target position'})
    display(signal_tail)

if trade_history.empty:
    print('No trade history available in window.')
else:
    trade_cols = ['trade_ts', 'signal_ts', 'regime_name', 'target_position', 'switched', 'target_ticker', 'account_balance', 'account_equity', 'hourly_pnl', 'hourly_pnl_pct', 'day_pnl', 'day_pnl_pct', 'execution_mode']
    available_cols = [c for c in trade_cols if c in trade_history.columns]
    trade_tail = trade_history[available_cols].sort_values('trade_ts', ascending=False).head(100).copy()
    for ts_col in ['trade_ts', 'signal_ts']:
        if ts_col in trade_tail.columns:
            trade_tail[ts_col] = trade_tail[ts_col].apply(fmt_timestamp)
    for text_col in ['regime_name', 'target_position', 'target_ticker', 'execution_mode']:
        if text_col in trade_tail.columns:
            trade_tail[text_col] = trade_tail[text_col].apply(_as_text)
    if 'switched' in trade_tail.columns:
        trade_tail['switched'] = trade_tail['switched'].apply(fmt_flag)
    for money_col in ['account_balance', 'account_equity']:
        if money_col in trade_tail.columns:
            trade_tail[money_col] = trade_tail[money_col].apply(fmt_money)
    for pnl_col in ['hourly_pnl', 'day_pnl']:
        if pnl_col in trade_tail.columns:
            trade_tail[pnl_col] = trade_tail[pnl_col].apply(fmt_signed_money)
    for pct_col in ['hourly_pnl_pct', 'day_pnl_pct']:
        if pct_col in trade_tail.columns:
            trade_tail[pct_col] = trade_tail[pct_col].apply(fmt_optional_percent)
    trade_tail = trade_tail.rename(columns={'trade_ts': 'Trade timestamp', 'signal_ts': 'Signal timestamp', 'regime_name': 'Regime', 'target_position': 'Target position', 'switched': 'Switched', 'target_ticker': 'Target ticker', 'account_balance': 'Account balance', 'account_equity': 'Account equity', 'hourly_pnl': 'Hourly pnl', 'hourly_pnl_pct': 'Hourly pnl %', 'day_pnl': 'Day pnl', 'day_pnl_pct': 'Day pnl %', 'execution_mode': 'Execution mode'})
    display(trade_tail)


displayHTML('<h2>8. Execution / health / freshness</h2>')
health_rows = [
    ('Dashboard freshness (min)', fmt_minutes(latest_feed.get('dashboard_data_freshness_minutes'))),
    ('Audit lag (min)', fmt_minutes(latest_health.get('audit_lag_minutes'))),
    ('Latest audit ts', fmt_timestamp(latest_feed.get('latest_audit_ts'))),
    ('Last execution action', _as_text(latest_feed.get('last_execution_action'))),
    ('Skip reason', _as_text(latest_feed.get('last_skip_reason'))),
    ('Error reason', _as_text(latest_feed.get('last_error_reason'))),
    ('Skip / 24h', _as_text(latest_feed.get('last_skip_count_24h', 0))),
    ('Error / 24h', _as_text(latest_feed.get('last_error_count_24h', 0))),
    ('Rate-limit / 24h', _as_text(latest_feed.get('last_rate_limit_count_24h', 0))),
]
display(pd.DataFrame(health_rows, columns=['Metric', 'Value']))

if not health_history.empty:
    health_tail = health_history[['rollup_ts', 'audit_lag_minutes', 'last_execution_action', 'last_skip_count_24h', 'last_error_count_24h', 'last_rate_limit_count_24h']].copy()
    health_tail['rollup_ts'] = health_tail['rollup_ts'].dt.strftime('%Y-%m-%d %H:%M')
    display(health_tail.tail(24))

